In [1]:
!git clone https://github.com/Shorstko/mai_python_2022
%cd /content/mai_python_2022/Homework

Cloning into 'mai_python_2022'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 94 (delta 20), reused 12 (delta 12), pack-reused 63
Unpacking objects: 100% (94/94), done.
/content/mai_python_2022/Homework


In [2]:
# =====================================
# ЗАДАНИЕ 1: Классы - декораторы
# =====================================
# TODO 1-1 - Добавить атрибут incidents типа список
# Возьмите код из предыдущего ДЗ
# Добавьте в класс MultirotorUAV атрибут incidents и внесите туда информацию обо всех найденных происшествиях для этой модели
# Не забудьте, что атрибут добавляется при помощи декоратора

In [28]:
import json
import csv
import re

class Aircraft:
	def __init__(self, weight):
		self._weight = weight

class UAV:
	def __init__(self):
		self._has_autopilot = True
		self._missions = []

	# напишите код для декоратора атрибута _missions
	@property
	def missions(self):
		return self._missions

	@missions.setter
	def missions(self, mission):
		self._missions.append(mission)

	# напишите публичный метод count_missions
	def count_missions(self):
		return len(self.missions[0])


class MultirotorUAV(Aircraft, UAV):
    def __init__(self, weight, model, brand):
        super().__init__(weight)
        UAV.__init__(self)
        self.__weight = weight
        self.__model = model
        self.__brand = brand
        self.__incidents = []

	# напишите публичный метод get_info
    def get_info(self):
        print(f'Масса {self.__weight}, Производитель {self.__brand}, Количество миссий {self.count_missions()}')

    # напишите публичный метод get_model
    def get_model(self):
        return self.__model

    @property
    def incidents(self):
        return self.__incidents
    
    @incidents.setter
    def incidents(self, incident):
        self.__incidents.append(incident)

    def add_incident(self, incident):
        self.__incidents.append(incident)
    
    def save_data(self):
        data = {"model" : self.get_model(), "mass" : self.__weight, "manufacturer" : self.__brand, "missions" : [], "incidents" : []}
        for x in self._missions:
            data['missions'].append(x)
        for x in self.__incidents:
            data['incidents'].append(x)
        with open(f'MultirotorUAV_{self.get_model()}.json', 'w') as f:
            json.dump(data, f)

with open("pilot_path.json") as f:
	json_data = json.load(f)
 
drone_catalog = {
	"DJI Mavic 2 Pro": {"weight":903, "brand":"DJI"},
	"DJI Mavic 2 Zoom": {"weight":900, "brand":"DJI"},
	"DJI Mavic 2 Enterprise Advanced": {"weight":920, "brand":"DJI"},
	"DJI Inspire 2": {"weight":1500, "brand":"DJI"},
	"DJI Mavic 3": {"weight":1000, "brand":"DJI"}
}

models = set()
for pilot in json_data:
    [models.add(mission["drone"]) for mission in json_data[pilot]["missions"]]

model_missions = dict()
for pilot in json_data:
  for mission in json_data[pilot]["missions"]:
    model_missions.setdefault(mission["drone"], [])
    model_missions[mission["drone"]] += [mission["mission"]]

drones_list = []
for i, model in enumerate(models):
    drones_list.append(MultirotorUAV(drone_catalog[model]["weight"], model, drone_catalog[model]["brand"]))
    drones_list[i].missions.append(model_missions[model])

In [29]:
# =====================================
# ЗАДАНИЕ 2: Файлы - CSV
# =====================================
# TODO 2-1 - Загрузите информацию об авиапроисшествиях из файла csv
# Проверьте по моделям (названия моделей возьмите из экземпляров класса MultirotorUAV), какие из них участвовали в авиапроисшествиях

# ВАШ КОД чтения данных из файла:

In [30]:
incidents_details = []
with open("faa_incidents.csv") as f:
    reader = csv.reader(f)
    lines = list(reader)
    incidents_details = [line[-1] for line in lines[1:]]

In [31]:
# =====================================
# ЗАДАНИЕ 3: Классы
# =====================================
# TODO 3-1 - Для каждой модели дрона добавьте в экземпляр класса информацию об авиапроисшествиях, в которых участвовала эта модель
# Информацию сохраните в атрибут incidents (используйте декораторы)
# Подсказка: вот так вы получаете названия модели для каждого экземпляра класса MultirotorUAV
# Экземпляры все так же находятся в списке (например, drones_cls_list)

In [32]:
for drone_cls in drones_list:
    drone = drone_cls.get_model()
    drone_name = ' '.join(drone.split(' ')[1:]).lower()

    for incident in incidents_details:
        if drone_name in incident.lower():
            drone_cls.add_incident(incident)

In [33]:
# TODO 3-2 - Добавьте в класс MultirotorUAV публичный метод save_data, который сохраняет статистику по дрону в файл
# Внимание! Метод save_data не принимает параметры. Название файла сформируйте как: название класса + название модели + расширение .json
# например: "MultirotorUAV_DJI Mavic 2 Pro.json"
# Подсказка: название класса вы можете получить вот так: self.__class__.__name__
# используйте ключи: "model", "weight", "brand", "missions", "incidents"
# например: {"model":"DJI Inspire 2", "weight": 1500, "info": "...", "manufacturer": "DJI", "missions": [], "incidents": []}

# ВАШ КОД - допишите код в объявлении класса

In [34]:
# =====================================
# ЗАДАНИЕ 4: Регулярные выражения
# =====================================
# TODO 4-1 - Выведите на экран собранную информацию по инцидентам по каждому дрону в таком виде:
# модель: инцидентов - количество
# 1 - краткий текст инцидента*
# полный текст инцидента
# * - краткий текст инцидента получайте следующим образом: в исходном тексте инцидента найдите модель, например, INSPIRE 2,
# и выведите все предложения, в которых встречается упоминание этой модели
# Подсказка 1: Полностью готовый код есть в лекции про регулярные выражения (пример про перелетных птиц).
# Ваши изменения: а) вставить вместо "зим" название модели дрона, б) поменять язык поиска на английский
# Подсказка 2: не забывайте использовать флаг re.I для игнорирования регистра символов
# Подсказка 3: перед тем, как искать, уберите из названия модели название производителя
# Подсказка 4: лучше не используйте re.compile. Для этого случая работает не очень

In [35]:
regex_pattern_fmt = r'[A-Z][^\.!?]+({})(?(1)[^\.!?]+[\.!?])'
drones_list = sorted(drones_list, key=lambda item: len(item.incidents))
for drone_cls in drones_list:
    drone_name = ' '.join(drone_cls.get_model().lower().split(' ')[1:]).strip()
    incidents_count = len(drone_cls.incidents)

    print(f"{drone_name}: инцидентов - {incidents_count}")
    if incidents_count == 0:
        continue

    pattern = regex_pattern_fmt.format(drone_name)
    for incident_idx, incident in enumerate(drone_cls.incidents):
        print(f'{incident_idx + 1} - ', end='')
        result = re.search(pattern, incident, flags=re.IGNORECASE)
        print(result.group())
        print(incident)

mavic 2 pro: инцидентов - 0
mavic 2 enterprise advanced: инцидентов - 0
mavic 3: инцидентов - 0
mavic 2 zoom: инцидентов - 1
1 - ON JULY 15, 2020 AT 1050 EDT, A DJI, MAVIC 2 ZOOM L1Z UAS, SERIAL # 0M6TG85R0A04ZP, UA FA REGISTRATION # FA3RE7RNWP, REGISTERED TO ^PRIVACY DATA OMITTED^ (PIC), REMOTE PILOT CERTIFICATE ^PRIVACY DATA OMITTED^, LOST CONTROLLED FLIGHT IN THE AREA OF ^PRIVACY DATA OMITTED^ AND HIT A BLACK NISSAN PICKUP TRUCK BEARING ^PRIVACY DATA OMITTED^ TRAVELING ALONG TAMIAMI TRAIL IN NORTH PORT CAUSING PROPERTY DAMAGE.
ON JULY 15, 2020 AT 1050 EDT, A DJI, MAVIC 2 ZOOM L1Z UAS, SERIAL # 0M6TG85R0A04ZP, UA FA REGISTRATION # FA3RE7RNWP, REGISTERED TO ^PRIVACY DATA OMITTED^ (PIC), REMOTE PILOT CERTIFICATE ^PRIVACY DATA OMITTED^, LOST CONTROLLED FLIGHT IN THE AREA OF ^PRIVACY DATA OMITTED^ AND HIT A BLACK NISSAN PICKUP TRUCK BEARING ^PRIVACY DATA OMITTED^ TRAVELING ALONG TAMIAMI TRAIL IN NORTH PORT CAUSING PROPERTY DAMAGE. THE UAS WAS FLOWN ON A RECREATIONAL FLIGHT OVER A CONSTRU

In [36]:
# TODO 4-2 - После вывода информации об инциденте сохраните всю информацию о дроне в файл .json при помощи метода save_data

In [37]:
for drone_cls in drones_list:
    drone_cls.save_data()